In [10]:
import json
import xml.etree.ElementTree as ET
import os
import re

# --- KONFIGURATION ---
json_input = '/Users/Elias/awg-app/src/assets/data/edition/series/1/section/5/m143/textcritics.json'
svg_folder = '/Users/Elias/awg-app/src/assets/img/edition/series/1/section/5/m143' 
output_folder = './output/'
prefix = "g-tkk-"

os.makedirs(output_folder, exist_ok=True)

def extract_numbers(text):
    """Extrahiert alle Ziffern aus einem String (z.B. 'M_143_TF1' -> '1431')"""
    return "".join(re.findall(r'\d+', str(text)))

# 1. JSON LADEN
with open(json_input, 'r', encoding='utf-8') as f:
    data = json.load(f)

all_entries = data.get('textcritics', data) if isinstance(data, dict) else data
available_svgs = [f for f in os.listdir(svg_folder) if f.endswith('.svg')]

# Status-Variablen
current_svg_tree = None
current_svg_elements = []
current_svg_ids_in_file = set()
current_mapping = {}
current_svg_filename = None

processed_output = []

print("--- Starte Verarbeitung nach Zahlen-Logik ---")

# 2. SEQUENTIELLE VERARBEITUNG
for entry in all_entries:
    if not isinstance(entry, dict):
        continue

    new_id = entry.get('id')
    
    if new_id:
        # Vorherige SVG speichern
        if current_svg_tree and current_svg_filename:
            current_svg_tree.write(os.path.join(output_folder, current_svg_filename), 
                                   encoding='utf-8', xml_declaration=True)

        # Neue ID-Zahlen extrahieren
        id_numbers = extract_numbers(new_id)
        
        # Passende SVG suchen (Zahlen-Vergleich)
        matching_file = next((f for f in available_svgs if id_numbers in extract_numbers(f)), None)
        
        if matching_file:
            current_svg_filename = matching_file
            print(f"\n📂 Anker: {new_id} (Zahlen: {id_numbers}) -> Match: {matching_file}")
            
            svg_path = os.path.join(svg_folder, matching_file)
            current_svg_tree = ET.parse(svg_path)
            root = current_svg_tree.getroot()
            
            current_svg_elements = []
            current_svg_ids_in_file = set()
            for elem in root.iter():
                if 'tkk' in (elem.get('class') or '').split():
                    oid = elem.get('id')
                    if oid:
                        current_svg_ids_in_file.add(oid)
                        current_svg_elements.append(elem)
            
            current_mapping = {}
        else:
            print(f"\n⚠️  Kein Match für ID {new_id} (Zahlen: {id_numbers})")
            current_svg_tree = None
            current_svg_filename = None

    # Verarbeitung der blockComments unter dem aktuellen Anker
    if current_svg_tree:
        comments_list = entry.get('commentary', {}).get('comments', [])
        for comment_group in comments_list:
            for b_comment in comment_group.get('blockComments', []):
                old_val = b_comment.get('svgGroupId')
                
                if old_val and old_val in current_svg_ids_in_file:
                    if old_val not in current_mapping:
                        current_mapping[old_val] = f"{prefix}{len(current_mapping) + 1}"
                    
                    new_val = current_mapping[old_val]
                    b_comment['svgGroupId'] = new_val
                    
                    # In der SVG sofort ändern
                    for elem in current_svg_elements:
                        if elem.get('id') == old_val:
                            elem.set('id', new_val)
                    
                    print(f"  ✅ {old_val} -> {new_val}")

    processed_output.append(entry)

# Letzte Datei speichern
if current_svg_tree and current_svg_filename:
    current_svg_tree.write(os.path.join(output_folder, current_svg_filename), 
                           encoding='utf-8', xml_declaration=True)

# 3. JSON SPEICHERN
with open(os.path.join(output_folder, 'textcritics_neu.json'), 'w', encoding='utf-8') as f:
    final_data = {"textcritics": processed_output} if isinstance(data, dict) and 'textcritics' in data else processed_output
    json.dump(final_data, f, indent=4, ensure_ascii=False)

print(f"\n🚀 Fertig! Dateien in '{output_folder}' gespeichert.")

--- Starte Verarbeitung nach Zahlen-Logik ---

📂 Anker: M_143_TF1 (Zahlen: 1431) -> Match: M143_Textfassung1-1von4-final.svg
  ✅ g570 -> g-tkk-1
  ✅ g571 -> g-tkk-2
  ✅ g600 -> g-tkk-3
  ✅ g601 -> g-tkk-4
  ✅ g629 -> g-tkk-5
  ✅ g630 -> g-tkk-6

🚀 Fertig! Dateien in './output/' gespeichert.
